## Data

In [2]:
import json
import pandas as pd
import numpy as np

import glob
import os

In [4]:
df = pd.read_csv("data/data_latest.csv")

df

,review_id,date,review_text,topic,subtopic,sentiment
0,1000087,2025-09-19,Вклад «Новые деньги» невозможно оформить без п...,Вклады,NaN,Negative
1,999494,2025-09-18,В июне 2025 года я порекомендовал премиальную ...,Дебетовые карты,NaN,Negative
2,999142,2025-09-17,Мошенниччиские аперации в интересах Ренессанс ...,Обслуживание,NaN,Negative
3,998360,2025-09-15,Купил услугу Газпром Бонус «Премиум» за 2 990 ...,Дебетовые карты,NaN,Negative
4,998516,2025-09-15,Производил оформление открытия срочного банков...,Вклады,«Накопительный»,Negative
...,...,...,...,...,...,...
4773,7470,2011-04-07,Ужастное обслуживание! Мало того потеряли доку...,Обслуживание,NaN,Negative
4774,7049,2011-03-28,Могут заблокировать рассчетную или кредитную к...,Кредитные карты,NaN,Negative
4775,5221,2011-01-25,"Мало того уже прошла неделя, а ПТС так и не ве...",Автокредиты,NaN,Negative
4776,5053,2011-01-16,Газпромбанк– отличный банк с отличными сотрудн...,Ипотека,NaN,Positive


### collecting synthetic dataset

In [5]:
json_paths = glob.glob("data/synthetic_dataset/output2/gemini*")

In [20]:
# json_paths_part = [
#     json_paths[-1],
#     json_paths[4]
# ]

synth_data = []
for path in json_paths:
    print(path)
    with open(path) as f:
        json_str = f.read()
        
        json_str = json_str.removeprefix("```json")
        if json_str[-3:] != "```":
            raise Exception(path)
        
        json_str = json_str.removesuffix("```")
        
        synth_data_range = json.loads(json_str)
        
    synth_data.extend(synth_data_range)

data/synthetic_dataset/output2\gemini_0-350.json
data/synthetic_dataset/output2\gemini_10150-10500.json
data/synthetic_dataset/output2\gemini_1050-1400.json
data/synthetic_dataset/output2\gemini_10500-10850.json
data/synthetic_dataset/output2\gemini_10850-11200.json
data/synthetic_dataset/output2\gemini_11200-11550.json
data/synthetic_dataset/output2\gemini_11550-11900.json
data/synthetic_dataset/output2\gemini_1400-1750.json
data/synthetic_dataset/output2\gemini_1750-2100.json
data/synthetic_dataset/output2\gemini_2100-2450.json
data/synthetic_dataset/output2\gemini_2450-2800.json
data/synthetic_dataset/output2\gemini_2800-3150.json
data/synthetic_dataset/output2\gemini_3150-3500.json
data/synthetic_dataset/output2\gemini_350-700.json
data/synthetic_dataset/output2\gemini_3500-3850.json
data/synthetic_dataset/output2\gemini_3850-4200.json
data/synthetic_dataset/output2\gemini_4200-4550.json
data/synthetic_dataset/output2\gemini_4550-4900.json
data/synthetic_dataset/output2\gemini_4900

In [22]:
ids = [int(synth_data[i]["id"]) for i in range(len(synth_data))]

In [23]:
set_ids = set(ids)

In [27]:
len(ids), len(set(ids))

(11791, 11654)

In [ ]:
df["review_id"].apply(lambda x: x in set_ids).sum() / df.shape[0]

np.int64(15)

In [29]:
with open('data/synthetic_dataset/synthetic_data_bankiru.json', 'w') as f:
    json.dump(synth_data, f)

### fixing broken dataset

In [159]:
not_done_ids = df["review_id"][df["review_id"].apply(lambda x: int(x) not in ids)].values

In [160]:
df_randomised = df.sample(df.shape[0], random_state=42)

In [162]:
df_randomised

,review_id,date,review_text,topic,subtopic,sentiment
33,989095,2025-08-19,Пользовался дебетовой картой газпромбанка пару...,Дебетовые карты,NaN,Negative
555,904370,2024-12-02,Здравствуйте! Газпромбанк — самый худший банк!...,Вклады,NaN,Positive
3413,408232,2021-08-11,Обратилась в банк с просьбой о выдаче автокред...,Автокредиты,NaN,Negative
2420,547546,2022-07-13,"Хороший банк, вчера вечером позвонил, консульт...",Кредитные карты,«Удобная»,Positive
4424,299984,2018-10-09,Ошибка подключения телекард на телефон. При на...,Обслуживание,NaN,Negative
...,...,...,...,...,...,...
4426,299718,2018-10-06,"Просто днищенские карты, дали карту для получе...",Дебетовые карты,NaN,Negative
466,912227,2024-12-23,"Заказала дебетовую карту Газпромбанк, привез к...",Дебетовые карты,«Умная карта»,Positive
3092,463192,2021-11-17,"Скачала мобильный банкинг ГПБ, чтобы наконец р...",Дистанционное обслуживание,NaN,Positive
3772,393190,2021-01-26,"Сняли 99р тихо, не знаю за что, код операции 1...",Ипотека,NaN,Positive


In [168]:
df_part_randomised = df_randomised.iloc[0:200].copy()

In [169]:
to_replace_ids = df_part_randomised["review_id"].values

In [175]:
this_is_absolytly_the_last_time_i_swear_set = set(not_done_ids.tolist() + to_replace_ids.tolist())

In [176]:
len(this_is_absolytly_the_last_time_i_swear_set)

679

In [177]:
df_to_redo = df[df["review_id"].apply(lambda x: x in this_is_absolytly_the_last_time_i_swear_set)]

In [178]:
df_to_redo

,review_id,date,review_text,topic,subtopic,sentiment
8,996657,2025-09-10,Прошлым летом оформил дебетовую карту Юнион пе...,Кредитные карты,NaN,Negative
15,993698,2025-09-03,Оформила договор на кредитную карту с льготным...,Кредитные карты,NaN,Positive
19,992313,2025-08-29,"Отвратительный банк, лишенный чести и совести,...",Дебетовые карты,NaN,Negative
23,992008,2025-08-28,"Хотела закрыть счёт кк, если быть точнее оформ...",Обслуживание,NaN,Positive
26,991359,2025-08-26,Исходный отзыв — касается ПАО Газпромбанк\n(гл...,Вклады,«Накопительный»,Negative
...,...,...,...,...,...,...
4718,58316,2013-03-14,12 марта с. Г. После много– кратных моих звонк...,Кредиты наличными,NaN,Negative
4740,44406,2012-05-26,"Прошла собеседование в данном банке, больше в ...",Кредиты наличными,NaN,Negative
4756,14156,2011-09-30,Я написал заявление отправил факсом только раз...,Кредитные карты,NaN,Negative
4774,7049,2011-03-28,Могут заблокировать рассчетную или кредитную к...,Кредитные карты,NaN,Negative


In [190]:
df_randomised = df_to_redo.sample(df_to_redo.shape[0], random_state=42)

In [199]:
df_randomised.shape[0]

679

In [201]:
df_start = 339
df_end = df_start + df_randomised.shape[0] // 2 + 1
df_start, df_end

(339, 679)

In [202]:
df_part_randomised = df_randomised.iloc[df_start : df_end].copy()

In [203]:
df_part_randomised["review_text"] = df_part_randomised["review_text"].str.replace("\n", " ")

In [204]:
texts = df_part_randomised.apply(lambda x: f"Отзыв {x['review_id']}: {x['review_text']}", axis=1)

In [205]:
combined_text = "\n\n".join(texts.values)

In [206]:
print(combined_text)

Отзыв 550568: Побывала в отделени на Рязанском проспекте и была приятно удивлена. Большой просторный офис, нет столпотворения, все сидят, кондиционер работает, есть столик для детей с раскрасками. Сам персонал вежливый, приветливый, да и между собой общаются уважительно, слышала, как старший коллега благодарил младшего за работу. Даже появилось желание поработаь там)

Отзыв 629780: Перед поездкой в ОАЭ заказала и в тот же день получила в отделении Газпромбанка мгновенную карту Union Pay, которая меня просто спасла за границей Я расплачивалась ею везде в аптеках, супермаркетах, ресторанах. За всю двухнедельную поездку ни одного отказа и да, деньги в банкомате я тоже с неё снимала. Выпуск карты стоил 5000 рублей, но благодаря спец. Условиям — повышенному кэшбеку, за первые 3 месяца карта себя окупила. Спасибо огромное за возможность чувствовать себя за границей надёжно и безопасно.

Отзыв 377461: Закончился срок действия карты 17.07.2020 В отделении Газпромбанка г. Мытищи ул. Мира 12/15 

In [207]:
with open(f'data/gemma_redo{df_start}-{df_end}.txt', 'w') as file:
    file.write(combined_text)

### creating dataset

In [3]:
from unsloth import FastModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Switching to PyTorch attention since your Xformers is broken.

Requires Flash-Attention version >=2.7.1,<=2.8.2 but got 2.8.3.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [5]:
max_seq_length = 4096

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-270m-it",
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    load_in_8bit = True, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

==((====))==  Unsloth 2025.9.8: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 3050 Laptop GPU. Num GPUs = 1. Max memory: 4.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
model = FastModel.get_peft_model(
    model,
    r = 128, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 128,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Making `model.base_model.model.model` require gradients


In [7]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma3",
)

In [8]:
print(df["review_text"][df["review_id"] == 573263].values)

['Особенно отмечу, насколько легко здесь завести карту. Оставил заявку на сайте ГПБ, с тобой связались и обсудили время и место доставки, курьер сам бесплатно привезет куда вам удобно. И все, никаких душных очередей в отдении. Если что-то спросить надо, курьер как правило все знает. Остальное можно подсмотреть в приложении, там все понятно, либо же набрать поддержке, они славные ребята. Работают на 5+. Сама карта тоже удобная, получаю по ней кешбек на заправках. В общем, лучше карты не встречала']


In [8]:
# synth_data

In [9]:
# {int(synth_data[i]["id"]) : synth_data[i]["topic_sentiment_pairs"] for i in range(10)}

In [4]:
SYSTEM_PROMPT = """\
Твоя задача — проанализировать отзывы клиентов Газпромбанка (ГПБ). Для каждого отзыва тебе необходимо:
1.  Определить все продукты/услуги (темы), о которых упоминает клиент.
2.  Для каждой упомянутой темы определить тональность высказывания: positive, negative или neutral.

**ВАЖНЫЕ ИНСТРУКЦИИ:**
-   **Темы:** В основном используй следующий список: Дебетовые карты, Офисное обслуживание, Дистанционное обслуживание, Кредиты наличными, Кредитные карты, Вклады, Ипотека, Автокредиты, Реструктуризация кредитов, Рефинансирование кредитов, Рефинансирование ипотеки, Обмен валют, Мобильное приложение, Потребительский кредит. Если в отзыве описана другая тема, ты можешь добавить ее. Помимо этого, можешь указывать уточняющие темы, такие как продукты банка, которые приведены ниже.
-   **Тональность:** `neutral` указывается тогда, когда тема упомянута как факт, без какой-либо эмоциональной окраски (ни положительной, ни отрицательной).
-   **Строгость:** Не выдумывай темы. Если в отзыве нет явного упоминания продукта или услуги, не включай его.
-   **Символы `****`:** Это либо конфиденциальные данные (номера телефонов), либо ненормативная лексика. Учитывай общий контекст вокруг них для определения тональности.
-   **Если темы нет:** Если в отзыве невозможно определить ни одну тему, верни пустой массив `topic_sentiment_pairs`.

Вот список всех продуктов банка на сегодняшний день:
* **дебетовые карты:**
    * Умная дебетовая карта «Мир»
    * Премиальная карта Mir Supreme
    * Дебетовая карта с кэшбэком для самозанятых
    * Умная дебетовая карта «Мир»
    * Карта для автолюбителей «Газпромбанк—Газпромнефть»
    * Виртуальная дебетовая карта ГПБ&ФК «Зенит»
    * Дебетовая Пенсионная карта
* **Кредитные карты:**
    * Кредитная карта с льготным периодом до 120 дней
    * Простая кредитная карта
    * Кредитная карта 90 дней
    * Кредитная карта 180 дней Премиум
    * Кредитная карта для самозанятых
* **Накопительные счета:**
    * Накопительный счет
    * «Ежедневная выгода»
    * «Ежедневный процент»
    * «Премиум»
    * Социальный счет
* **Вклады:**
    * Вклад «Новые деньги»
    * Вклад «Ключевой момент»
    * Вклад «Копить»
    * Вклад «В Плюсе»
    * Вклад «Расширяй возможности»
    * Социальный вклад
* **Кредиты:**
    * Кредит наличными
    * Кредит наличными под залог недвижимости
    * Кредит на авто и другие цели
    * Рефинансирование потребительских кредитов
    * Дачный кредит
    * Кредит на образование
    * Кредит наличными для бюджетников
* **Другие услуги банка:**
    * Газпромбанк Мобайл
    * Газпромбанк Travel
    * Gazprom Pay
    * GorodPay
    * Газпром Бонус
    * Страховые и сервисные продукты
    * Депозитарные услуги


**Формат ответа — строго JSON:**
{
    "topic_sentiment_pairs": [
        {
            "topic": "Название темы 1",
            "sentiment": "positive/negative/neutral"
        },
        {
            "topic": "Название темы 2",
            "sentiment": "positive/negative/neutral"
        }
    ]
}


**Примеры:**
Отзыв 1: Мобильное приложение просто ужасное, постоянно вылетает. А вот вклад оформил недавно — условия хорошие.

Ответ:
{
    "topic_sentiment_pairs": [
        {
            "topic": "Мобильное приложение",
            "sentiment": "negative"
        },
        {
            "topic": "Вклады",
            "sentiment": "positive"
        }
    ]
}


Отзыв 2: Звонил узнать насчет ипотеки, но в поддержке ничем не помогли.

Ответ:
{
    "topic_sentiment_pairs": [
        {
            "topic": "Дистанционное обслуживание",
            "sentiment": "negative"
        },
        {
            "topic": "Ипотека",
            "sentiment": "neutral"
        }
    ]
},

**Вот отзывы клиентов, которые необходимо обработать:**
"""

In [10]:
# # Сдеалть разделение на train/test по дате

# from torch.utils.data import Dataset

# class ReviewsDataset(Dataset):
#     def __init__(self, topics_sentiments_json, original_reviews_csv):
#         with open(topics_sentiments_json) as f:
#             topics_sentiments_full = json.load(f)
            
#         self.topics_sentiments_pair = {
#             int(topics_sentiments_full[i]["id"]) : topics_sentiments_full[i]["topic_sentiment_pairs"] 
#             for i in range(len(topics_sentiments_full))
#         }
            
#         self.original_reviews_df = pd.read_csv(original_reviews_csv)

#     def __len__(self):
#         return len(self.original_reviews_df.shape[0])

#     def __getitem__(self, idx):
#         original_review_series = self.original_reviews_df.iloc[idx]
#         review_id = original_review_series["review_id"]
        
#         user_prompt = original_review_series["review_text"]
        
#         assistant_answer = str(self.topics_sentiments_pair[review_id])
        
#         return {"user_prompt" : user_prompt, "assistant_answer" : assistant_answer}

In [ ]:
# Сдеалть разделение на train/test по дате

from datasets import Dataset

topics_sentiments_json = "mount/data/synthetic_data.json"
original_reviews_csv = "mount/data/data_latest.csv"

with open(topics_sentiments_json) as f:
    topics_sentiments_full = json.load(f)
    
topics_sentiments_pair = {
    int(topics_sentiments_full[i]["id"]) : topics_sentiments_full[i]["topic_sentiment_pairs"] 
    for i in range(len(topics_sentiments_full))
}

original_reviews_df = pd.read_csv(original_reviews_csv)


user_prompts = []
assistant_answers = []

for i in range(original_reviews_df.shape[0]):
    original_review_series = original_reviews_df.iloc[i]
    review_id = original_review_series["review_id"]

    user_prompt = original_review_series["review_text"]

    assistant_answer = str(topics_sentiments_pair[review_id])
    
    user_prompts.append(user_prompt)
    assistant_answers.append(assistant_answer)
    

dataset_dict = {"user_prompt" : user_prompts, "assistant_answer" : assistant_answers}

dataset = Dataset.from_dict(dataset_dict)

In [17]:
dataset = dataset.train_test_split(test_size=0.1)

dataset_train = dataset["train"]
dataset_test = dataset["test"]

In [20]:
dataset_train[100]

{'user_prompt': 'Подала документы на рефинансирования сказали, что по документам все проходит, через 7 дней позвонили и сказали, чтобы мне сделали рефинансирования я должна погасить предыдущую карту, а это 169000, простите где я их должна взять? Для этого человек и подаёт на рефинансирования и чтоб был процент меньше, менеджер банка вообще позвонила и в конце бросила трубку, это не правильно, почему нельзя сразу взять и с нового кредита перекрыть старый. Такого ужаса ещё не в одном банке не предлагали. Прошу пересмотреть и сделать рефинансирования нормально.',
 'assistant_answer': "[{'topic': 'Рефинансирование кредитов', 'sentiment': 'negative'}, {'topic': 'Кредитные карты', 'sentiment': 'negative'}]"}

In [21]:
def convert_to_chatml(example):
    return {
        "conversations": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["user_prompt"]},
            {"role": "assistant", "content": example["assistant_answer"]}
        ]
    }

dataset_train = dataset_train.map(
    convert_to_chatml
)

dataset_test = dataset_test.map(
    convert_to_chatml
)

Map:   0%|          | 0/4300 [00:00<?, ? examples/s]

Map:   0%|          | 0/478 [00:00<?, ? examples/s]

In [22]:
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
    return { "text" : texts, }

dataset_train = dataset_train.map(formatting_prompts_func, batched = True)

dataset_test = dataset_test.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/4300 [00:00<?, ? examples/s]

Map:   0%|          | 0/478 [00:00<?, ? examples/s]

## training

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_train,
    eval_dataset = dataset_test, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 1, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 1, # Set this for 1 full training run.
        # max_steps = 100,
        learning_rate = 5e-5, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir="outputs",
        report_to = "none", # Use this for WandB etc
        dataset_num_proc=1,
        do_eval=True,
    ),
)

Unsloth: Tokenizing ["text"]:   0%|          | 0/4300 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"]:   0%|          | 0/478 [00:00<?, ? examples/s]

In [24]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

Map (num_proc=12):   0%|          | 0/4300 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/478 [00:00<?, ? examples/s]

In [25]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

'<bos><start_of_turn>user\nТвоя задача — проанализировать отзывы клиентов Газпромбанка (ГПБ). Для каждого отзыва тебе необходимо:\n1.  Определить все продукты/услуги (темы), о которых упоминает клиент.\n2.  Для каждой упомянутой темы определить тональность высказывания: positive, negative или neutral.\n\n**ВАЖНЫЕ ИНСТРУКЦИИ:**\n-   **Темы:** В основном используй следующий список: Дебетовые карты, Офисное обслуживание, Дистанционное обслуживание, Кредиты наличными, Кредитные карты, Вклады, Ипотека, Автокредиты, Реструктуризация кредитов, Рефинансирование кредитов, Рефинансирование ипотеки, Обмен валют, Мобильное приложение, Потребительский кредит. Если в отзыве описана другая тема, ты можешь добавить ее. Помимо этого, можешь указывать уточняющие темы, такие как продукты банка, которые приведены ниже.\n-   **Тональность:** `neutral` указывается тогда, когда тема упомянута как факт, без какой-либо эмоциональной окраски (ни положительной, ни отрицательной).\n-   **Строгость:** Не выдумывай

In [26]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

"                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [27]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,300 | Num Epochs = 1 | Total steps = 1,075
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 1 x 1) = 4
 "-____-"     Trainable parameters = 30,375,936 of 298,474,112 (10.18% trained)


Step,Training Loss
1,0.921200
2,1.096500
3,0.603500
4,0.857200
5,0.370400
6,0.147800
7,0.411800
8,0.275800
9,0.362600
10,0.304500


Unsloth: Will smartly offload gradients to save VRAM!


In [ ]:
from transformers import TextStreamer

In [111]:
test_idx = 56

print(dataset_test[test_idx]["user_prompt"])

messages = [
    [
        {'role': 'system','content' : SYSTEM_PROMPT},
        {"role" : 'user', 'content' : dataset_test[test_idx]["user_prompt"]}
    ]
]
text = tokenizer.apply_chat_template(
    messages[0],
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
).removeprefix('<bos>')

Лет 5 назад был положительный опыт кредитования здесь.
Теперь же взял дебетовую карту на зарплатный проект. Все оформил онлайн. Карту привезли быстро, работает четко. Вопросов нет.


In [135]:
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 125,
    temperature = 0.5, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

print(dataset_test[test_idx]["conversations"][-1]["content"])

[{'topic': 'Кредиты наличными', 'sentiment': 'positive'}, {'topic': 'Дебетовые карты', 'sentiment': 'positive'}]<end_of_turn>
[{'topic': 'Дебетовые карты', 'sentiment': 'positive'}, {'topic': 'Дистанционное обслуживание', 'sentiment': 'positive'}]


In [ ]:
model.save_pretrained("mount/models/gemma-3-270m-it-revews-fine-tune-v1")  # Local saving
tokenizer.save_pretrained("mount/models/gemma-3-270m-it-revews-fine-tune-v1")

('mount/models/gemma-3-270m-it-revews-fine-tune-v1/tokenizer_config.json',
 'mount/models/gemma-3-270m-it-revews-fine-tune-v1/special_tokens_map.json',
 'mount/models/gemma-3-270m-it-revews-fine-tune-v1/chat_template.jinja',
 'mount/models/gemma-3-270m-it-revews-fine-tune-v1/tokenizer.model',
 'mount/models/gemma-3-270m-it-revews-fine-tune-v1/added_tokens.json',
 'mount/models/gemma-3-270m-it-revews-fine-tune-v1/tokenizer.json')

## Running on the whole dataset

In [29]:
from unsloth import FastLanguageModel
import torch
from itertools import islice
from math import floor

from tqdm.autonotebook import tqdm

In [30]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "mount/models/gemma-3-270m-it-revews-fine-tune-v1", # YOUR MODEL YOU USED FOR TRAINING
    max_seq_length = 2048,
    load_in_4bit = False,
    load_in_8bit = True
)

==((====))==  Unsloth 2025.8.10: Fast Gemma3 patching. Transformers: 4.55.4. vLLM: 0.9.2.
   \\   /|    NVIDIA GeForce RTX 3050 Laptop GPU. Num GPUs = 1. Max memory: 4.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.6. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
!ls .cache/huggingface/hub

blobs  refs  snapshots


In [ ]:
# model.save_pretrained_merged("gemma3-270m-it-reviews-v1", tokenizer, save_method = "merged_16bit",)
model.push_to_hub_merged("JosephThePatrician/gemma3-270m-it-reviews-v1", tokenizer, save_method = "merged_16bit", token = "token")

In [ ]:
model.push_to_hub_merged("JosephThePatrician/gemma3-270m-it-reviews-4bit-v1", tokenizer, save_method = "merged_4bit", token = "token")

In [7]:
model = FastLanguageModel.for_inference(model)

In [18]:
def batched(iterable, n):
    "Batch data into lists of length n. The last batch may be shorter."
    # batched('ABCDEFG', 3) --> ABC DEF G
    it = iter(iterable)
    while True:
        batch = tuple(islice(it, n))
        if not batch:
            return
        yield batch


def make_data_batch(reviews):
    messages = [
        [
            {'role': 'system','content' : SYSTEM_PROMPT},
            {"role" : 'user', 'content' : review}
        ]
        for review in reviews
    ]
    
    texts = tokenizer.apply_chat_template(
        messages,
        tokenize = False,
        add_generation_prompt = True, # Must add for generation
    )
    
    texts = [text.removeprefix('<bos>') for text in texts]
    
    return texts


def predict_batch(reviews, batch_size=2):
    answers = []
    for review_batch in tqdm(batched(reviews, batch_size), total=floor(len(reviews) / batch_size)):
        
        # print(len(review_batch))
        # print(review_batch)
        
        texts = make_data_batch(review_batch)
        
        # print(len(texts))
        # print(texts)
        
        tokens = tokenizer(texts, padding_side="left", padding=True, return_tensors = "pt").to("cuda")
        
        out_tokens = model.generate(
            **tokens,
            max_new_tokens = 256,
            temperature = 0.1, top_p = 0.95, top_k = 64,
            # streamer = TextStreamer(tokenizer, skip_prompt = True),
        )
        
        answer_batch = tokenizer.batch_decode(out_tokens, skip_special_tokens=True)
        answer_batch = [answer_batch[i].split("\nmodel\n")[1] for i in range(len(answer_batch))]
        answers.extend(answer_batch)
    
    return answers

In [13]:
reviews = df["review_text"].values[:50]

In [16]:
predicted_topic_sentiment_pairs = predict_batch(reviews, 12)

  0%|          | 0/4 [00:00<?, ?it/s]

In [12]:
len(predicted_topic_sentiment_pairs)

4778

In [29]:
df.iloc[200]

review_id                                                 948975
date                                                  2025-04-17
review_text    11.04.25Г примерно в 12:30, я, пришёл в ТЦ «Тр...
topic                                               Обслуживание
subtopic                                                     NaN
sentiment                                               Negative
Name: 200, dtype: object

In [32]:
df["review_text"].values[100]

'Отрицательный опыт общения. Полное разочарование. Никому не посоветую открывать вклад в этом банке. Отвратительное обслуживание.'

In [33]:
predicted_topic_sentiment_pairs[100]

"[{'topic': 'Вклады', 'sentiment': 'negative'}, {'topic': 'Офисное обслуживание', 'sentiment': 'negative'}]"

In [16]:
with open("mount/data/predictions_v1.txt", mode="w") as f:
    f.write(str(predicted_topic_sentiment_pairs))

In [42]:
df_structured = pd.read_csv("mount/data/reviews_stuctured.csv")
df_structured

,reviewId,idSiteSpecific,source,date,review_text,topic,subtopic,rating
0,0,1000087,sravni.ru,2025-09-19,Вклад «Новые деньги» невозможно оформить без п...,Вклады,NaN,NaN
1,1,999494,sravni.ru,2025-09-18,В июне 2025 года я порекомендовал премиальную ...,Дебетовые карты,NaN,NaN
2,2,999142,sravni.ru,2025-09-17,Мошенниччиские аперации в интересах Ренессанс ...,Обслуживание,NaN,NaN
3,3,998360,sravni.ru,2025-09-15,Купил услугу Газпром Бонус «Премиум» за 2 990 ...,Дебетовые карты,NaN,NaN
4,4,998516,sravni.ru,2025-09-15,Производил оформление открытия срочного банков...,Вклады,«Накопительный»,NaN
...,...,...,...,...,...,...,...,...
4773,4773,7470,sravni.ru,2011-04-07,Ужастное обслуживание! Мало того потеряли доку...,Обслуживание,NaN,NaN
4774,4774,7049,sravni.ru,2011-03-28,Могут заблокировать рассчетную или кредитную к...,Кредитные карты,NaN,NaN
4775,4775,5221,sravni.ru,2011-01-25,"Мало того уже прошла неделя, а ПТС так и не ве...",Автокредиты,NaN,NaN
4776,4776,5053,sravni.ru,2011-01-16,Газпромбанк– отличный банк с отличными сотрудн...,Ипотека,NaN,NaN


In [43]:
df_review_topics = pd.read_csv("mount/data/reviews_topics.csv")
df_review_topics

,id,reviewId,topicId,sentiment
0,0,0,0,Negative
1,1,1,1,Negative
2,2,2,0,Negative
3,3,3,17,Negative
4,4,4,17,Negative
...,...,...,...,...
4773,4773,4773,2,Negative
4774,4774,4774,4,Negative
4775,4775,4775,8,Negative
4776,4776,4776,17,Positive


In [64]:
df_topics_info = pd.read_csv("mount/data/topics_info.csv")
df_topics_info

,id,name,description
0,0,Вклады,NaN
1,1,Дебетовые карты,NaN
2,2,Обслуживание,NaN
3,3,Дистанционное обслуживание,NaN
4,4,Кредитные карты,NaN
5,5,Кредиты наличными,NaN
6,6,Другие услуги,NaN
7,7,Обмен валют,NaN
8,8,Ипотека,NaN
9,9,Автокредиты,NaN


In [40]:
predicted_topic_sentiment_pairs[0]

"[{'topic': 'Вклады', 'sentiment': 'negative'}, {'topic': 'Страховые и сервисные продукты', 'sentiment': 'negative'}]"

In [56]:
ids = []
topics = []
sentiments = []

for i, pairs_str in enumerate(predicted_topic_sentiment_pairs):
    # topics_sentiments_pairs = review["topic_sentiment_pairs"]
    review_id = df_structured["reviewId"][i]
    # print(pairs_str)
    pairs = eval(pairs_str)
    for pair in pairs:
        ids.append(int(review_id))
        topics.append(pair["topic"])
        sentiments.append(pair["sentiment"])

In [75]:
unique_topics_sorted = pd.Series(topics).value_counts().index.values

unique_topics_ids = list(range(len(unique_topics_sorted)))

topics_ids_dict = {
    unique_topics_sorted[i] : unique_topics_ids[i] for i in range(len(unique_topics_sorted))
}

df_topics_info_v1 = pd.DataFrame(
    {
        "id" : unique_topics_ids,
        "name" : unique_topics_sorted,
        "description" : None
    }
)

# df_topics_info_v1

In [77]:
df_review_topics

,id,reviewId,topicId,sentiment
0,0,0,0,Negative
1,1,1,1,Negative
2,2,2,0,Negative
3,3,3,17,Negative
4,4,4,17,Negative
...,...,...,...,...
4773,4773,4773,2,Negative
4774,4774,4774,4,Negative
4775,4775,4775,8,Negative
4776,4776,4776,17,Positive


In [86]:
topicids = [topics_ids_dict[topics[i]] for i in range(len(topics))]

In [90]:
df_topics_sentiments_v1 = pd.DataFrame(
    {
        "id" : list(range(len(ids))),
        "reviewId" : ids,
        "topicId" : topicids,
        "sentiment" : sentiments
    }
)

df_topics_sentiments_v1

,id,reviewId,topicId,sentiment
0,0,0,6,negative
1,1,0,7,negative
2,2,1,0,negative
3,3,1,17,negative
4,4,1,1,negative
...,...,...,...,...
8948,8948,4775,0,negative
8949,8949,4776,4,positive
8950,8950,4776,2,positive
8951,8951,4777,3,negative


In [91]:
df_topics_info_v1.to_csv("mount/data/structured_data/topics_info.csv", index=False)

df_topics_sentiments_v1.to_csv("mount/data/structured_data/reviews_topics_v1.csv", index=False)